In [3]:

# 4. Out-of-Sample Evaluation

#All strategy parameters were selected using 2020–2023 development data.

#The final specification was locked before evaluation on untouched 2024–2025 data.


# Step 1: Locked strategy specification
# Parameters selected using the development sample only.
# No parameter tuning is allowed during out-of-sample validation.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locked strategy parameters
FORMATION_H = 336          # 14-day residual momentum signal
HOLDING_H = 168            # 7-day holding / rebalance period
N_LONG = 2
N_SHORT = 2
COST_BPS = 7

# Development / validation split
DEV_START = "2020-01-01"
DEV_END = "2023-12-31"

VAL_START = "2024-01-01"
VAL_END = "2025-12-31"

print("Locked strategy specification")
print("-----------------------------")
print("Signal: Residual Momentum")
print("Formation window:", FORMATION_H, "hours")
print("Holding period:", HOLDING_H, "hours")
print("Long / Short:", N_LONG, "/", N_SHORT)
print("Transaction cost:", COST_BPS, "bps")
print("Development:", DEV_START, "to", DEV_END)
print("Validation:", VAL_START, "to", VAL_END)

Locked strategy specification
-----------------------------
Signal: Residual Momentum
Formation window: 336 hours
Holding period: 168 hours
Long / Short: 2 / 2
Transaction cost: 7 bps
Development: 2020-01-01 to 2023-12-31
Validation: 2024-01-01 to 2025-12-31


In [4]:
# Step 2: Load full dataset for out-of-sample validation

df = pd.read_csv(
    "crypto_hourly_clean_2020_2025.csv",
    parse_dates=["open_time"]
)

df = (
    df
    .sort_values(["symbol", "open_time"])
    .reset_index(drop=True)
)

return_panel_full = (
    df
    .pivot(
        index="open_time",
        columns="symbol",
        values="ret_1h"
    )
    .sort_index()
)

print("Full return panel shape:", return_panel_full.shape)

print(
    "Full period:",
    return_panel_full.index.min(),
    "to",
    return_panel_full.index.max()
)

print(
    "Assets:",
    len(return_panel_full.columns)
)

display(return_panel_full.head())

Full return panel shape: (52577, 8)
Full period: 2020-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
Assets: 8


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2020-01-01 00:00:00+00:00,-0.002131,-0.001312,-0.002531,-0.002583,-0.002245,-0.001980,-0.000484,-0.002436
2020-01-01 01:00:00+00:00,0.006406,0.007402,0.005469,0.005577,0.013735,0.010942,0.008236,0.006390
2020-01-01 02:00:00+00:00,0.005456,0.003500,0.003683,0.004803,0.001607,0.009702,0.005526,0.002426
2020-01-01 03:00:00+00:00,-0.004221,-0.002224,-0.002463,-0.009610,-0.004968,-0.003999,-0.007646,-0.001081
2020-01-01 04:00:00+00:00,-0.001211,-0.001795,-0.001071,0.001642,0.000000,0.002008,0.001445,-0.001753


In [5]:
# Step 2.1: Check out-of-sample period

oos_return_panel = return_panel_full.loc[
    (return_panel_full.index >= VAL_START) &
    (return_panel_full.index < "2026-01-01")
].copy()

print(
    "OOS period:",
    oos_return_panel.index.min(),
    "to",
    oos_return_panel.index.max()
)

print(
    "OOS shape:",
    oos_return_panel.shape
)

print(
    "Rows with all 8 assets available:",
    oos_return_panel.notna().all(axis=1).sum()
)

print(
    "Missing observations by asset:"
)

display(
    oos_return_panel.isna().sum()
)

OOS period: 2024-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
OOS shape: (17544, 8)
Rows with all 8 assets available: 17544
Missing observations by asset:


symbol
ADAUSDT     0
BNBUSDT     0
BTCUSDT     0
DOGEUSDT    0
ETHUSDT     0
LINKUSDT    0
LTCUSDT     0
XRPUSDT     0
dtype: int64

In [6]:
# Step 3: Construct equal-weight crypto market factor

market_return_full = return_panel_full.mean(axis=1)

print("Market return observations:",
      market_return_full.notna().sum())

print(
    "Market period:",
    market_return_full.index.min(),
    "to",
    market_return_full.index.max()
)

display(
    market_return_full.describe()
)

Market return observations: 52562
Market period: 2020-01-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00


count    52562.000000
mean         0.000103
std          0.008458
min         -0.185252
25%         -0.003125
50%          0.000280
75%          0.003700
max          0.121130
dtype: float64

In [7]:
# Step 3.1: Check OOS market factor

market_return_oos = market_return_full.loc[
    (market_return_full.index >= VAL_START) &
    (market_return_full.index <= VAL_END + " 23:59:59")
]

print("OOS market observations:",
      len(market_return_oos))

display(
    market_return_oos.describe()
)

OOS market observations: 17544


count    17544.000000
mean         0.000057
std          0.007354
min         -0.109977
25%         -0.002966
50%          0.000220
75%          0.003390
max          0.085618
dtype: float64

In [8]:
# Step 4: Estimate rolling beta

BETA_WINDOW = 720
BETA_MIN_OBS = int(BETA_WINDOW * 0.95)   # 684

market_var = (
    market_return_full
    .rolling(
        window=BETA_WINDOW,
        min_periods=BETA_MIN_OBS
    )
    .var()
)

beta_full = pd.DataFrame(
    index=return_panel_full.index,
    columns=return_panel_full.columns,
    dtype=float
)

for asset in return_panel_full.columns:

    rolling_cov = (
        return_panel_full[asset]
        .rolling(
            window=BETA_WINDOW,
            min_periods=BETA_MIN_OBS
        )
        .cov(market_return_full)
    )

    beta_full[asset] = rolling_cov / market_var


print(
    "Valid beta rows:",
    beta_full.notna().all(axis=1).sum()
)

display(
    beta_full.tail()
)

Valid beta rows: 51894


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2025-12-31 19:00:00+00:00,1.252184,0.668382,0.739533,1.139558,1.077417,1.284099,0.937290,0.901538
2025-12-31 20:00:00+00:00,1.253333,0.668518,0.739227,1.140164,1.077531,1.284417,0.936098,0.900712
2025-12-31 21:00:00+00:00,1.253310,0.668342,0.737648,1.140526,1.076541,1.285290,0.936834,0.901510
2025-12-31 22:00:00+00:00,1.253290,0.668048,0.737798,1.140058,1.076842,1.285650,0.936901,0.901412
2025-12-31 23:00:00+00:00,1.253267,0.668041,0.737783,1.140064,1.076884,1.285699,0.936900,0.901363


In [9]:
# Step 4.1: Calculate residual returns

market_component_full = (
    beta_full.mul(
        market_return_full,
        axis=0
    )
)

residual_return_full = (
    return_panel_full
    - market_component_full
)

print(
    "Valid residual rows:",
    residual_return_full.notna().all(axis=1).sum()
)

display(
    residual_return_full.tail()
)

Valid residual rows: 51879


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2025-12-31 19:00:00+00:00,-0.002064,0.003500,-0.000715,0.001996,0.000253,-0.000056,-0.000805,-0.002110
2025-12-31 20:00:00+00:00,0.002334,-0.001803,0.000441,-0.002375,0.000163,0.000832,0.000778,-0.000370
2025-12-31 21:00:00+00:00,-0.001856,0.001118,-0.000401,-0.000066,-0.000671,-0.000996,-0.000944,0.003816
2025-12-31 22:00:00+00:00,-0.001093,-0.000901,0.000127,0.002990,-0.000337,-0.003234,0.001458,0.000990
2025-12-31 23:00:00+00:00,0.002376,0.000258,-0.000398,-0.001074,-0.001096,-0.001554,0.000263,0.001226


In [10]:
# Step 5: Construct locked 336h residual momentum signal

SIGNAL_WINDOW = FORMATION_H       # 336
SIGNAL_MIN_OBS = int(SIGNAL_WINDOW * 0.95)

signal_336_full = (
    residual_return_full
    .rolling(
        window=SIGNAL_WINDOW,
        min_periods=SIGNAL_MIN_OBS
    )
    .sum()
)

print("Signal window:", SIGNAL_WINDOW, "hours")
print("Minimum observations:", SIGNAL_MIN_OBS)

print(
    "Valid signal rows:",
    signal_336_full.notna().all(axis=1).sum()
)

display(signal_336_full.tail())

Signal window: 336 hours
Minimum observations: 319
Valid signal rows: 51575


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2025-12-31 19:00:00+00:00,-0.085884,0.033808,0.025981,-0.060719,0.065394,0.014296,0.017283,-0.010159
2025-12-31 20:00:00+00:00,-0.081703,0.030802,0.028526,-0.064828,0.065480,0.019422,0.018588,-0.016286
2025-12-31 21:00:00+00:00,-0.083121,0.032797,0.023719,-0.062126,0.061071,0.016139,0.021288,-0.009767
2025-12-31 22:00:00+00:00,-0.082748,0.030098,0.023719,-0.060132,0.061218,0.013848,0.019903,-0.005905
2025-12-31 23:00:00+00:00,-0.079955,0.029833,0.023207,-0.060242,0.058877,0.010259,0.021239,-0.003218


In [11]:
# Step 5.1: Extract OOS signals

signal_336_oos = signal_336_full.loc[
    (signal_336_full.index >= VAL_START) &
    (signal_336_full.index <= VAL_END + " 23:59:59")
].copy()

print("OOS signal observations:",
      len(signal_336_oos))

print(
    "OOS rows with all 8 signals available:",
    signal_336_oos.notna().all(axis=1).sum()
)

display(signal_336_oos.head())

OOS signal observations: 17544
OOS rows with all 8 signals available: 17544


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2024-01-01 00:00:00+00:00,-0.055076,0.233235,-0.010145,-0.111432,0.000664,0.003342,-0.021678,-0.038910
2024-01-01 01:00:00+00:00,-0.053141,0.236277,-0.008298,-0.115108,0.000708,-0.003098,-0.020269,-0.037071
2024-01-01 02:00:00+00:00,-0.046695,0.221914,-0.006623,-0.110575,-0.001634,0.001013,-0.017014,-0.040387
2024-01-01 03:00:00+00:00,-0.047838,0.222572,-0.008667,-0.107280,-0.002723,-0.000145,-0.017926,-0.037994
2024-01-01 04:00:00+00:00,-0.050560,0.224643,-0.008221,-0.108882,0.001390,0.000620,-0.018358,-0.040631


In [12]:
# Step 6: Define OOS rebalance dates
# Rebalance every 168 hours (7 days)

valid_signal_oos = signal_336_oos.notna().all(axis=1)

rebalance_dates_oos = (
    signal_336_oos
    .index[valid_signal_oos][::HOLDING_H]
)

print("Number of OOS rebalances:",
      len(rebalance_dates_oos))

print("First 5 rebalance dates:")
print(rebalance_dates_oos[:5])

print("\nLast 5 rebalance dates:")
print(rebalance_dates_oos[-5:])

Number of OOS rebalances: 105
First 5 rebalance dates:
DatetimeIndex(['2024-01-01 00:00:00+00:00', '2024-01-08 00:00:00+00:00',
               '2024-01-15 00:00:00+00:00', '2024-01-22 00:00:00+00:00',
               '2024-01-29 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='open_time', freq=None)

Last 5 rebalance dates:
DatetimeIndex(['2025-12-01 00:00:00+00:00', '2025-12-08 00:00:00+00:00',
               '2025-12-15 00:00:00+00:00', '2025-12-22 00:00:00+00:00',
               '2025-12-29 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='open_time', freq=None)


In [13]:
# Step 6.1: Construct locked Long 2 / Short 2 portfolio

weights_oos = pd.DataFrame(
    0.0,
    index=rebalance_dates_oos,
    columns=signal_336_oos.columns
)

for date in rebalance_dates_oos:

    signal_today = signal_336_oos.loc[date]

    # Highest residual momentum
    long_assets = signal_today.nlargest(N_LONG).index

    # Lowest residual momentum
    short_assets = signal_today.nsmallest(N_SHORT).index

    weights_oos.loc[date, long_assets] = 0.25
    weights_oos.loc[date, short_assets] = -0.25


print("OOS portfolio formations:",
      len(weights_oos))

display(weights_oos.head())

OOS portfolio formations: 105


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2024-01-01 00:00:00+00:00,-0.25,0.25,0.00,-0.25,0.00,0.25,0.00,0.00
2024-01-08 00:00:00+00:00,-0.25,0.25,0.25,-0.25,0.00,0.00,0.00,0.00
2024-01-15 00:00:00+00:00,-0.25,0.00,0.00,-0.25,0.25,0.25,0.00,0.00
2024-01-22 00:00:00+00:00,0.00,0.00,-0.25,0.00,0.00,0.25,0.25,-0.25
2024-01-29 00:00:00+00:00,0.00,0.25,0.00,0.00,-0.25,0.25,0.00,-0.25


In [14]:
# Step 7: Construct future 168h compounded returns

future_168_full = (
    (1 + return_panel_full)
    .rolling(window=HOLDING_H)
    .apply(np.prod, raw=True)
    .shift(-HOLDING_H)
    - 1
)

future_168_oos = future_168_full.reindex(
    weights_oos.index
)

print("Future return observations:",
      future_168_oos.notna().all(axis=1).sum())

display(future_168_oos.head())

Future return observations: 104


symbol,ADAUSDT,BNBUSDT,BTCUSDT,DOGEUSDT,ETHUSDT,LINKUSDT,LTCUSDT,XRPUSDT
open_time,,,,,,,,
2024-01-01 00:00:00+00:00,-0.177956,-5.279898e-02,0.027234,-0.131805,-0.043960,-0.134837,-0.127566,-0.062480
2024-01-08 00:00:00+00:00,0.084842,2.014775e-02,-0.030992,0.036158,0.137155,0.144692,0.106056,0.012117
2024-01-15 00:00:00+00:00,-0.053263,5.167874e-02,-0.014719,0.045539,-0.013416,0.032318,0.030071,-0.062938
2024-01-22 00:00:00+00:00,-0.027932,-4.475743e-02,0.007537,-0.069476,-0.083245,-0.060737,-0.061691,-0.047454
2024-01-29 00:00:00+00:00,0.002853,4.440892e-15,0.009281,-0.006868,0.010340,0.235246,-0.017171,-0.038896


In [15]:
# Step 7.1: OOS gross strategy return

gross_return_oos = (
    weights_oos * future_168_oos
).sum(axis=1)

gross_return_oos = gross_return_oos.dropna()

print("OOS gross periods:",
      len(gross_return_oos))

display(
    gross_return_oos.describe()
)

OOS gross periods: 105


count    105.000000
mean       0.008705
std        0.040069
min       -0.076556
25%       -0.012958
50%        0.002720
75%        0.025904
max        0.178745
dtype: float64

In [16]:
# Step 8: OOS turnover

turnover_oos = (
    weights_oos
    .diff()
    .abs()
    .sum(axis=1)
)

turnover_oos.iloc[0] = (
    weights_oos.iloc[0].abs().sum()
)

print("Average OOS turnover:",
      round(turnover_oos.mean(), 4))

print("Median OOS turnover:",
      round(turnover_oos.median(), 4))

Average OOS turnover: 1.0524
Median OOS turnover: 1.0


In [17]:
# Step 8.1: Apply locked 7 bps transaction cost

cost_rate = COST_BPS / 10000

transaction_cost_oos = (
    turnover_oos * cost_rate
)

net_return_oos = (
    gross_return_oos
    - transaction_cost_oos.loc[gross_return_oos.index]
)

print("Average transaction cost:",
      round(transaction_cost_oos.mean(), 6))

display(
    net_return_oos.describe()
)

Average transaction cost: 0.000737


count    105.000000
mean       0.007968
std        0.040090
min       -0.077606
25%       -0.013734
50%        0.001670
75%        0.024854
max        0.178045
dtype: float64

In [18]:
# Step 9: OOS net performance

r = net_return_oos.dropna()

periods_per_year = 365 * 24 / HOLDING_H

annual_return_oos = (
    r.mean() * periods_per_year
)

annual_vol_oos = (
    r.std() * np.sqrt(periods_per_year)
)

sharpe_oos = (
    annual_return_oos / annual_vol_oos
    if annual_vol_oos > 0
    else np.nan
)

equity_oos = (
    1 + r
).cumprod()

running_max_oos = (
    equity_oos.cummax()
)

max_drawdown_oos = (
    equity_oos / running_max_oos - 1
).min()

print("OOS Net Annualised Return:",
      round(annual_return_oos, 4))

print("OOS Net Annualised Volatility:",
      round(annual_vol_oos, 4))

print("OOS Net Sharpe:",
      round(sharpe_oos, 4))

print("OOS Maximum Drawdown:",
      round(max_drawdown_oos, 4))

OOS Net Annualised Return: 0.4155
OOS Net Annualised Volatility: 0.2895
OOS Net Sharpe: 1.4352
OOS Maximum Drawdown: -0.1991


# 成功了
- The strategy remained profitable out of sample, with a net Sharpe of 1.44 after 7 bps transaction costs, compared with 1.06 during development.”

In [20]:
# Step 10: OOS yearly performance

yearly_results = []

for year in [2024, 2025]:

    r_year = net_return_oos[
        net_return_oos.index.year == year
    ].dropna()

    n = len(r_year)

    annual_return = (
        r_year.mean() * periods_per_year
    )

    annual_vol = (
        r_year.std() * np.sqrt(periods_per_year)
    )

    sharpe = (
        annual_return / annual_vol
        if annual_vol > 0 else np.nan
    )

    equity = (1 + r_year).cumprod()

    max_drawdown = (
        equity / equity.cummax() - 1
    ).min()

    yearly_results.append({
        "year": year,
        "n_periods": n,
        "annual_return": annual_return,
        "annual_vol": annual_vol,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown
    })

yearly_oos = pd.DataFrame(yearly_results)

display(yearly_oos.round(4))

,year,n_periods,annual_return,annual_vol,sharpe,max_drawdown
0,2024,53,0.6086,0.3430,1.7744,-0.1991
1,2025,52,0.2187,0.2224,0.9834,-0.1618


In [21]:
%whos


Variable                Type             Data/Info
--------------------------------------------------
BETA_MIN_OBS            int              684
BETA_WINDOW             int              720
COST_BPS                int              7
DEV_END                 str              2023-12-31
DEV_START               str              2020-01-01
FORMATION_H             int              336
HOLDING_H               int              168
N_LONG                  int              2
N_SHORT                 int              2
SIGNAL_MIN_OBS          int              319
SIGNAL_WINDOW           int              336
VAL_END                 str              2025-12-31
VAL_START               str              2024-01-01
annual_return           float64          0.2186998718084564
annual_return_oos       float64          0.41548270293509665
annual_vol              float64          0.22240008741964695
annual_vol_oos          float64          0.289487242747539
asset                   str              XRPUSDT
b

In [41]:

import statsmodels.api as sm

# OOS net strategy returns
r = net_return_oos.dropna()

# Constant-only regression: test whether mean return = 0
X = np.ones(len(r))

model = sm.OLS(r.values, X).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 1}
)

mean_weekly_return = r.mean()
t_stat = model.tvalues[0]
p_value = model.pvalues[0]

print("Observations:", len(r))
print("Mean weekly net return:", mean_weekly_return)
print("Mean weekly net return (%):", mean_weekly_return * 100)
print("Newey-West t-stat:", t_stat)
print("p-value:", p_value)

Observations: 105
Mean weekly net return: 0.007968161426152538
Mean weekly net return (%): 0.7968161426152538
Newey-West t-stat: 1.9019041663586767
p-value: 0.05718368441860633


In [43]:
# Save OOS net return series for validation notebook
net_return_oos.to_csv("net_return_oos.csv", header=["net_return"])

In [45]:
print("Saved net_return_oos.csv")
print(net_return_oos.head())

Saved net_return_oos.csv
open_time
2024-01-01 00:00:00+00:00    0.029831
2024-01-08 00:00:00+00:00   -0.033311
2024-01-15 00:00:00+00:00    0.005957
2024-01-22 00:00:00+00:00   -0.021678
2024-01-29 00:00:00+00:00    0.065250
dtype: float64
